# SAE Absorption Generalizes Beyond Spelling — Numeric & Taxonomic (Gemma Scope L12)

**Two-track CCRG generality experiment (C3).** Single sparse-autoencoder (SAE) latents suffer
from *feature absorption*: a general "parent" latent silently stops firing on a specialised
sub-context because a second latent has absorbed that slice. Absorption has been documented
almost only for **first-letter spelling**. This experiment asks: **does it generalise to
non-spelling token hierarchies (numbers, countries)?**

The method treats SAE latents as a learned knowledge representation and builds **cluster-level
units** instead of trusting a single latent:

1. **Anchor** — pick the general "parent" latent from content-flip pairs (non-circular).
2. **Non-triviality gate (GO/NO-GO)** — does the anchor leave *holes* (sub-contexts it misses)
   that a *mutually-exclusive*, sub-context-precise latent covers?
3. **K-track anchored greedy set-cover** — grow a small, human-auditable unit
   `{anchor + named specialist absorbers}` that closes the holes.
4. **Baselines** — compare the unit against the raw anchor latent, marginal-attribution oracle
   pools `(g)`/`(h)` (SCR/TPP-style), and a **non-SAE dense probe** (logistic / diff-of-means on
   raw residuals) — with paired bootstrap, exact McNemar, and Holm multiplicity.

**This demo** runs the full downstream methodology on a curated **~100-row subset** of the
numeric testbed. The heavy Phase-0/1 encoding (a `gemma-2-2b` forward pass + JumpReLU SAE encode)
was run **once** by the original `method.py`; its cached, max-pooled SAE latents and residuals are
embedded per row in `mini_demo_data.json`, so the notebook needs **no GPU** and finishes in
seconds. Latent IDs are re-derived on the subset (so they differ from the paper), and the paired
C3 comparison is under-powered at 100 rows — we report that honestly and show how to scale back up.


In [ ]:
# --- Dependencies (works on Google Colab and locally) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab -> always install
_pip('loguru==0.7.3')

# Core scientific packages: pre-installed on Colab (do NOT reinstall there — it corrupts the
# loaded C extensions); install locally at Colab's exact versions to mirror the environment.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1',
         'statsmodels==0.14.6', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (from the original method.py + matplotlib for the demo figure) ---
import json, os, sys, time
from pathlib import Path
import numpy as np
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from loguru import logger
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# --- Load mini_demo_data.json (GitHub raw URL, with local fallback for offline runs) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-2/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("metadata:")
for k, v in data["metadata"].items():
    print(f"  {k}: {v}")
print(f"\nloaded {len(data['rows'])} rows")

## Configuration

All tunable parameters live here. They are set to **small demo values** so the notebook runs
in seconds on the ~100-row subset; the commented `# paper:` values reproduce the full run.
The *scientific* gate thresholds (`G1_RECALL`, `JACCARD_MAX`, ...) are kept at the paper's
pinned values — only the **scale** knobs (bootstrap repeats, eligibility cutoff) are reduced.

In [ ]:
# ---- scale knobs (reduced for the demo) ----
SEED              = 20240617
N_MIN_ELIGIBLE    = 5      # paper: 150  (min diagnostic positives for a sub-context to be "eligible")
B_GATE            = 500    # paper: 2000 (bootstrap reps for gate / greedy coverage CIs)
B_PAIRED          = 1000   # paper: 10000 (paired-bootstrap reps for unit-minus-baseline CIs)
B_NULL            = 200    # paper: 1000 (reps for sign-flip / random-k / surface-invariance nulls)

# ---- scientific thresholds (pinned from the paper; kept fixed) ----
G1_RECALL         = 0.60   # gate G1: anchor (parent) recall floor
JACCARD_MAX       = 0.10   # mutual-exclusivity ceiling (firing-Jaccard vs members)
SUBCTX_PREC       = 0.70   # an absorber must concentrate in one sub-context
GAIN_MIN          = 0.05   # min marginal hole-coverage gain
PRECISION_FLOOR   = 0.70   # content-flip precision floor for any unit member
GREEDY_MAX_MEMBERS = 8     # max latents in a K-track unit

# ---- shapes / bookkeeping ----
WIDTH    = data["metadata"]["width"]      # 16384 (Gemma Scope width_16k)
D_MODEL  = data["metadata"]["d_model"]    # 2304  (gemma-2-2b residual dim)
HIERARCHY = data["metadata"]["hierarchy"] # "numeric"

rng = np.random.default_rng(SEED)
RESULTS_DIR = Path("demo_results"); RESULTS_DIR.mkdir(exist_ok=True)
_GLOBAL = {}   # would hold the SAE decoder W_dec; intentionally absent in this no-GPU demo
print(f"width={WIDTH} d_model={D_MODEL} hierarchy={HIERARCHY}")

## Phase 0/1 — precomputed SAE encodings

The original pipeline runs each row's text through `google/gemma-2-2b`, takes the residual at
`blocks.12.hook_resid_post` (HF `hidden_states[13]`, FVU-selected), and encodes it with the
Gemma Scope **JumpReLU** SAE loaded straight from DeepMind's `params.npz`. The SAE forward is
shown below for reference (it defines the math; it is *not* executed here because it needs the
GPU model). For the demo we load the **max-pooled latents** (`lat_idx`/`lat_val`, sparse) and the
**mean-pooled residual** (`resid`, 2304-d) that were cached during the original run.

In [ ]:
class JumpReLUSAE:
    """Gemma Scope JumpReLU SAE (canonical DeepMind forward; no sae_lens dependency).
        encode(x) = relu(x @ W_enc + b_enc) * (x @ W_enc + b_enc > threshold)
        decode(a) = a @ W_dec + b_dec
    Shown for reference — the demo uses the cached encodings it produced."""
    def __init__(self, params_path, device):
        import numpy as _np
        d = _np.load(params_path)
        self.W_enc = d["W_enc"]; self.W_dec = d["W_dec"]
        self.b_enc = d["b_enc"]; self.b_dec = d["b_dec"]
        self.threshold = d["threshold"]; self.device = device

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        return np.where(pre > self.threshold, np.maximum(pre, 0.0), 0.0)

    def decode(self, a):
        return a @ self.W_dec + self.b_dec

Reconstruct the sparse latent matrix `lat` (`[N, width]`, CSR) and the residual matrix
`resid` (`[N, 2304]`) from the embedded per-row encodings — exactly the two arrays the original
`encode_rows` returned.

In [ ]:
rows = data["rows"]
N = len(rows)

# CSR sparse latents, assembled strictly in row order (== original encode_rows output)
_d, _i, _p = [], [], [0]
for r in rows:
    _i += r["lat_idx"]; _d += r["lat_val"]; _p.append(len(_i))
lat = sp.csr_matrix((np.array(_d, np.float32), np.array(_i, np.int32), np.array(_p, np.int64)),
                    shape=(N, WIDTH))
resid = np.array([r["resid"] for r in rows], dtype=np.float16)   # mean-pooled residuals
logger.info(f"reconstructed lat {lat.shape} nnz={lat.nnz} | resid {resid.shape} | "
            f"mean nnz/row={lat.nnz/N:.0f}")

## Statistical & evaluation helpers (verbatim from `method.py`)

Bootstrap CIs, paired-difference bootstrap, exact McNemar, Holm-Bonferroni multiplicity, ROC-AUC,
and a recall-matched threshold.

In [ ]:
def bootstrap_ci(values, B=2000, alpha=0.05):
    """Bootstrap mean CI over a 1-D array of per-item values."""
    if len(values) == 0:
        return 0.0, 0.0, 0.0
    idx = rng.integers(0, len(values), size=(B, len(values)))
    means = values[idx].mean(1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(values.mean()), float(lo), float(hi)


def paired_diff_ci(a, b, B=10000, alpha=0.05):
    """Paired bootstrap CI on mean(a-b) where a,b are per-item 0/1 hit vectors (same items)."""
    d = a.astype(np.float64) - b.astype(np.float64)
    n = len(d)
    if n == 0:
        return 0.0, 0.0, 0.0
    idx = rng.integers(0, n, size=(B, n))
    means = d[idx].mean(1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(d.mean()), float(lo), float(hi)


def mcnemar_exact(a, b):
    """Exact McNemar p-value on paired 0/1 hits (all items are positives -> hit==correct)."""
    from statsmodels.stats.contingency_tables import mcnemar
    a = a.astype(bool); b = b.astype(bool)
    n00 = int((~a & ~b).sum()); n01 = int((~a & b).sum())
    n10 = int((a & ~b).sum()); n11 = int((a & b).sum())
    tbl = [[n11, n10], [n01, n00]]
    try:
        return float(mcnemar(tbl, exact=True).pvalue)
    except Exception:
        return float("nan")


def holm(pvals):
    """Holm-Bonferroni corrected p-values keyed identically to input."""
    from statsmodels.stats.multitest import multipletests
    keys = [k for k, v in pvals.items() if v == v]  # drop nan
    if not keys:
        return {k: float("nan") for k in pvals}
    raw = [pvals[k] for k in keys]
    _, corr, _, _ = multipletests(raw, method="holm")
    out = {k: float(c) for k, c in zip(keys, corr)}
    for k in pvals:
        out.setdefault(k, float("nan"))
    return out


def auc(scores, labels):
    if len(np.unique(labels)) < 2:
        return 0.5
    return float(roc_auc_score(labels, scores))


def match_threshold(scores, target_recall):
    """Threshold on a positive-only score vector achieving the target recall (fraction >= tau)."""
    if len(scores) == 0:
        return 0.0
    q = np.quantile(scores, max(0.0, 1.0 - target_recall))
    return float(q)


def recovered_ge1(unit):
    return len(unit) >= 2

## Admission test — signature-K & surface invariance (verbatim)

`signature-K`: does the *pooled* unit beat its best single member on content-flip AUC by more than
an AUC-matched random-k null? `surface invariance`: does the unit respond to the **concept**, not
its surface form (within-pair surface response not above a cross-pair null)?

In [ ]:
def admission_check(unit, anchor, cr_idx, A_on, A_off, lat, meta, rt, role, fold, pair_id, sub, width):
    """signature-K: pooled-max content-response AUC(unit) - best-single-member AUC vs AUC-matched
    random-k null; + unit-level surface invariance on surface pairs."""
    Npair = A_on.shape[0]
    y = np.concatenate([np.ones(Npair), np.zeros(Npair)])  # x_on=1, x_off=0

    def member_pool_auc(pool):
        on = A_on[:, pool].max(1) if len(pool) > 1 else A_on[:, pool[0]]
        off = A_off[:, pool].max(1) if len(pool) > 1 else A_off[:, pool[0]]
        return auc(np.concatenate([on, off]), y)

    pooled_auc = member_pool_auc(unit)
    single_aucs = [member_pool_auc([m]) for m in unit]
    best_single = max(single_aucs)
    gainK = pooled_auc - best_single
    k = len(unit); B = B_NULL
    null_gains = []
    pool_src = cr_idx if len(cr_idx) >= k else np.arange(width)
    for _ in range(B):
        rp = rng.choice(pool_src, size=k, replace=False).tolist()
        pa = member_pool_auc(rp)
        bs = max(member_pool_auc([m]) for m in rp)
        null_gains.append(pa - bs)
    null95 = float(np.percentile(null_gains, 95))
    sigK = bool(gainK > null95) and recovered_ge1(unit)

    sp_mask = (rt == "surface_pair")
    sa, sb = {}, {}
    for i in np.where(sp_mask)[0]:
        pid = pair_id[i]
        if role[i] == "surface_a":
            sa[pid] = i
        elif role[i] == "surface_b":
            sb[pid] = i
    spids = [p for p in sa if p in sb]
    surf_inv = {"n_surface_pairs": len(spids)}
    if spids and len(unit) > 0:
        ai = np.array([sa[p] for p in spids]); bi = np.array([sb[p] for p in spids])
        amat = np.asarray(lat[ai][:, unit].todense()).max(1)
        bmat = np.asarray(lat[bi][:, unit].todense()).max(1)
        obs = float(np.mean(np.abs(amat - bmat)))
        null = []
        if len(spids) > 2:
            for _ in range(B_NULL):
                perm = rng.permutation(len(spids))
                null.append(float(np.mean(np.abs(amat - bmat[perm]))))
        null_mean = float(np.mean(null)) if null else obs
        null05 = float(np.percentile(null, 5)) if null else obs
        null95s = float(np.percentile(null, 95)) if null else obs
        surf_inv.update({"observed_surface_response": obs, "null_mean": null_mean,
                         "null_05": null05, "null_95": null95s,
                         "surface_invariant": bool(obs <= null95s)})
    return {"pooled_auc": float(pooled_auc), "best_single_auc": float(best_single),
            "gainK": float(gainK), "null_95": null95, "signature_K_pass": sigK,
            "surface_invariance": surf_inv}

## Phase 7 — form-free KG edge agreement (verbatim, decoder-gated)

A *corroborating* diagnostic that scores each `anchor -> absorber` edge by the projection-based
`absorption_fraction`. It needs the **full SAE decoder** `W_dec` (`[16384, 2304]`), which is too
large to embed in the lightweight demo — so it cleanly returns a "skipped" note. The core
contribution (gate / K-track / baselines / admission) does **not** depend on it. `get_eligible`
lists the frozen, model-independent sub-contexts per hierarchy.

In [ ]:
def formfree_edge_agreement(edges, anchor, CR, lat, resid, diagpos_rows, sub_diagpos,
                            anchor_fire, d_p_unit, width):
    """For each edge anchor->absorber(l*,s): on diag hole-tokens of sub-context s where the anchor
    is silent, diag_absorber = argmax over CR latents of absorption_fraction (~ enc_l * W_dec_l . d_p).
    EDGE AGREEMENT = fraction where diag_absorber == l* (top1/top3). NULL = random CR latent."""
    if not edges:
        return {"edges": [], "note": "no edges proposed"}
    if "W_dec" not in _GLOBAL:
        return {"edges": edges,
                "note": "KG-agreement diagnostic requires the full SAE decoder (W_dec); "
                        "skipped in the lightweight (no-GPU) demo"}
    sae_wdec = _GLOBAL["W_dec"]
    wdec_dp = sae_wdec @ d_p_unit
    lat_dp = lat[diagpos_rows].tocsr()
    CRset = np.asarray(CR)
    sub_lat = np.asarray(lat_dp[:, CRset].todense())
    weighted = sub_lat * wdec_dp[CRset][None, :]
    weighted[sub_lat <= 0] = -np.inf
    diag_arg = weighted.argmax(1)
    has_fire = np.isfinite(weighted.max(1))
    diag_absorber_latent = np.where(has_fire, CRset[diag_arg], -1)
    top3 = np.argsort(-weighted, axis=1)[:, :3]
    top3_latent = CRset[top3]
    out_edges = []
    for e in edges:
        lstar = e["absorber"]; s = e["specializes"]
        m = (sub_diagpos == s) & (~anchor_fire)
        idx = np.where(m)[0]
        if len(idx) == 0:
            out_edges.append({**e, "n_hole_tokens": 0, "agreement_top1": None}); continue
        top1 = float((diag_absorber_latent[idx] == lstar).mean())
        t3 = float(np.any(top3_latent[idx] == lstar, axis=1).mean())
        nulls = []
        firing_idx = idx[has_fire[idx]]
        for _ in range(B_NULL):
            rl = int(rng.choice(CRset))
            nulls.append(float((diag_absorber_latent[firing_idx] == rl).mean()) if len(firing_idx) else 0.0)
        out_edges.append({**e, "n_hole_tokens": int(len(idx)), "agreement_top1": top1,
                          "agreement_top3": t3, "null_mean": float(np.mean(nulls)),
                          "null_95": float(np.percentile(nulls, 95)),
                          "above_null": bool(top1 > np.percentile(nulls, 95))})
    rates = [e["agreement_top1"] for e in out_edges if e.get("agreement_top1") is not None]
    return {"edges": out_edges, "mean_agreement_top1": float(np.mean(rates)) if rates else None,
            "mean_null": float(np.mean([e["null_mean"] for e in out_edges if "null_mean" in e])) if rates else None}


def get_eligible(name):
    if name == "numeric":
        return ["year", "percent", "currency", "date", "decimal", "integer", "comma_number", "ordinal"]
    return ["Canada", "France", "Iran", "United States", "Brazil", "Georgia", "New Zealand", "Spain",
            "Mexico", "China", "Japan", "India", "Italy", "Ireland", "Australia", "Poland", "Germany",
            "United Kingdom", "Israel", "Russia"]

## Core pipeline — `analyze_hierarchy` (verbatim from `method.py`)

This is the heart of the method, run end-to-end on one hierarchy:
**Phase 1** build content-flip / corpus index sets · **Phase 2** content-responsive prefilter +
**anchor** · **Phase 3** non-triviality **gate** + threshold sensitivity · **Phase 4** non-SAE
parent probe (disjoint fold) · **Phase 5** K-track anchored greedy set-cover + admission ·
**Phase 6** `(g)`/`(h)` marginal-attribution baselines + sliced recall + paired stats ·
**Phase 7** form-free KG agreement.

The only changes vs. the original are that the bootstrap repeat counts and the eligibility cutoff
read from the **config cell** (the demo's small values) instead of being hard-coded.

In [ ]:
def analyze_hierarchy(name, rows, lat, resid, eligible, width):
    logger.info(f"=== analyzing hierarchy '{name}' (rows={len(rows)}, width={width}) ===")
    meta = rows
    rt = np.array([r["metadata_row_type"] for r in meta])
    role = np.array([r["metadata_pair_role"] for r in meta])
    fold = np.array([r["metadata_fold"] for r in meta])
    label = np.array([1 if r["output"] == "positive" else 0 for r in meta])
    sub = np.array([r["metadata_sub_context"] for r in meta], dtype=object)
    pair_id = np.array([r["metadata_pair_id"] for r in meta], dtype=object)
    lat = lat.tocsr(); lat.eliminate_zeros()
    cp_train = (rt == "content_pair") & (fold == "train")
    pairs = {}
    for i in np.where(cp_train)[0]:
        pid = pair_id[i]; pairs.setdefault(pid, {})[role[i]] = i
    pair_ids = [p for p, d in pairs.items() if "x_on" in d and "x_off" in d]
    on_idx = np.array([pairs[p]["x_on"] for p in pair_ids])
    off_idx = np.array([pairs[p]["x_off"] for p in pair_ids])
    pair_sub = np.array([sub[pairs[p]["x_on"]] for p in pair_ids], dtype=object)
    Npair = len(pair_ids); logger.info(f"  content pairs (train): {Npair}")
    corp_train = (rt == "corpus") & (fold == "train")
    corp_diag = (rt == "corpus") & (fold == "diagnostic")
    diag_pos = corp_diag & (label == 1)
    diag_rows = np.where(corp_diag)[0]; diagpos_rows = np.where(diag_pos)[0]
    logger.info(f"  corpus train={int(corp_train.sum())} diag={int(corp_diag.sum())} diag_pos={len(diagpos_rows)}")
    A_on = np.asarray(lat[on_idx].todense()); A_off = np.asarray(lat[off_idx].todense())
    R = A_on - A_off; FIRE_on = A_on > 0; FIRE_off = A_off > 0
    mean_R = R.mean(0); B_null = B_NULL
    signs = rng.integers(0, 2, size=(Npair, B_null)) * 2 - 1
    null_means = (R.T @ signs) / Npair
    tau95 = np.percentile(null_means, 95, axis=1)
    content_responsive = (mean_R > tau95) & (mean_R > 0)
    cr_idx = np.where(content_responsive)[0]
    logger.info(f"  content-responsive latents: {len(cr_idx)} / {width}")
    fires_on_cnt = FIRE_on.sum(0)
    precision_l = np.divide((FIRE_on & ~FIRE_off).sum(0), np.maximum(fires_on_cnt, 1), dtype=np.float64)
    cover_count = (FIRE_on & (R > 0)).sum(0)
    neg_rows = np.where(corp_train & (label == 0))[0]
    neg_fire_rate = np.asarray((lat[neg_rows] > 0).mean(0)).ravel() if len(neg_rows) else np.zeros(width)
    anchor_pool = cr_idx[(precision_l[cr_idx] >= PRECISION_FLOOR) & (neg_fire_rate[cr_idx] < 0.5)]
    if len(anchor_pool) == 0:
        anchor_pool = cr_idx[precision_l[cr_idx] >= PRECISION_FLOOR]
    if len(anchor_pool) == 0:
        anchor_pool = cr_idx
    anchor = int(anchor_pool[np.argmax(cover_count[anchor_pool])])
    anchor_cover = set(np.where(FIRE_on[:, anchor] & (R[:, anchor] > 0))[0].tolist())
    anchor_recall_cf = len(anchor_cover) / max(Npair, 1)
    anchor_fire_diagpos = np.asarray((lat[diagpos_rows][:, anchor] > 0).todense()).ravel()
    anchor_recall_corp = float(anchor_fire_diagpos.mean()) if len(diagpos_rows) else 0.0
    logger.info(f"  ANCHOR latent={anchor} | precision={precision_l[anchor]:.3f} "
                f"| recall_cf={anchor_recall_cf:.3f} | recall_corp={anchor_recall_corp:.3f} "
                f"| neg_fire={neg_fire_rate[anchor]:.3f}")
    from sklearn.linear_model import LogisticRegression
    tr_rows = np.where(corp_train)[0]
    Xtr = resid[tr_rows].astype(np.float32); ytr = label[tr_rows]
    probe = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced"); probe.fit(Xtr, ytr)
    d_p = probe.coef_[0].astype(np.float64); d_p_unit = d_p / (np.linalg.norm(d_p) + 1e-9)
    Xdiag = resid[diagpos_rows].astype(np.float32)
    probe_pred = probe.predict(Xdiag)
    parent_probe_recall_by_sub = {}; sub_diagpos = sub[diagpos_rows]
    for s in sorted(set(sub_diagpos.tolist())):
        m = sub_diagpos == s
        if m.sum() >= 1:
            parent_probe_recall_by_sub[str(s)] = float(probe_pred[m].mean())
    overall_probe_recall = float(probe_pred.mean()) if len(diagpos_rows) else 0.0
    logger.info(f"  parent probe (non-SAE) overall diag recall={overall_probe_recall:.3f}")
    CR = cr_idx
    fire_diagpos = (lat[diagpos_rows][:, CR] > 0)
    fire_diagpos_d = np.asarray(fire_diagpos.todense())
    anchor_in_cr = np.where(CR == anchor)[0]; anchor_fire = anchor_fire_diagpos
    holes_corp = ~anchor_fire; H0 = int(holes_corp.sum())
    holes_cf = np.array([p not in anchor_cover for p in range(Npair)])
    logger.info(f"  holes: corpus={H0}/{len(diagpos_rows)} content-flip={int(holes_cf.sum())}/{Npair}")
    inter = (fire_diagpos_d & anchor_fire[:, None]).sum(0).astype(np.float64)
    union = fire_diagpos_d.sum(0) + anchor_fire.sum() - inter
    jaccard_cr = np.divide(inter, np.maximum(union, 1))
    subs_sorted = sorted([s for s in set(sub_diagpos.tolist()) if s is not None])
    fire_per_sub = np.zeros((len(subs_sorted), len(CR)))
    for si, s in enumerate(subs_sorted):
        m = sub_diagpos == s
        if m.any():
            fire_per_sub[si] = fire_diagpos_d[m].sum(0)
    tot_fire = fire_diagpos_d.sum(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        subctx_prec_cr = np.where(tot_fire > 0, fire_per_sub.max(0) / np.maximum(tot_fire, 1), 0.0)
        subctx_arg_cr = fire_per_sub.argmax(0)
    hole_cov_cr = fire_diagpos_d[holes_corp].mean(0) if H0 > 0 else np.zeros(len(CR))
    def gain_ci_low(latpos):
        col = fire_diagpos_d[holes_corp, latpos].astype(np.float64)
        _, lo, _ = bootstrap_ci(col, B=B_GATE); return lo
    cand_mask = (np.arange(len(CR)) != (anchor_in_cr[0] if len(anchor_in_cr) else -1))
    passing = []
    for j in np.where(cand_mask)[0]:
        if (jaccard_cr[j] < JACCARD_MAX and subctx_prec_cr[j] >= SUBCTX_PREC
                and hole_cov_cr[j] >= GAIN_MIN and precision_l[CR[j]] >= PRECISION_FLOOR):
            lo = gain_ci_low(j)
            if lo > 0:
                passing.append({"latent": int(CR[j]), "jaccard": float(jaccard_cr[j]),
                    "subctx_precision": float(subctx_prec_cr[j]),
                    "specializes": str(subs_sorted[int(subctx_arg_cr[j])]) if subs_sorted else None,
                    "hole_coverage_gain": float(hole_cov_cr[j]), "gain_ci_low": float(lo),
                    "content_precision": float(precision_l[CR[j]])})
    passing.sort(key=lambda d: -d["hole_coverage_gain"])
    anchor_recall_best = max(anchor_recall_cf, anchor_recall_corp)
    g1 = anchor_recall_best >= G1_RECALL; g2 = len(passing) >= 1; gate_pass = bool(g1 and g2)
    logger.info(f"  GATE: G1(recall>={G1_RECALL})={g1} (best={anchor_recall_best:.3f}) | "
                f"G2(>=1 absorber)={g2} (n_passing={len(passing)}) | PASS={gate_pass}")
    threshold_sensitivity = {}
    for jt in (0.05, 0.10, 0.20):
        for pt in (0.60, 0.70, 0.80):
            for gt in (0.03, 0.05, 0.10):
                cnt = int(np.sum((jaccard_cr[cand_mask] < jt) & (subctx_prec_cr[cand_mask] >= pt)
                                 & (hole_cov_cr[cand_mask] >= gt)
                                 & (precision_l[CR[cand_mask]] >= PRECISION_FLOOR)))
                threshold_sensitivity[f"J<{jt}_P>={pt}_G>={gt}"] = cnt
    unit = [anchor]; member_fire = {anchor: anchor_fire}; H = holes_corp.copy(); edges = []
    H0_fixed = max(int(holes_corp.sum()), 1)
    while H.sum() > 0 and len(unit) < GREEDY_MAX_MEMBERS:
        best_l, best_gain, best_j = None, 0.0, None
        cover_H = (fire_diagpos_d & H[:, None]); gains = cover_H.sum(0).astype(np.float64) / H0_fixed
        order = np.argsort(-gains)
        for j in order:
            if gains[j] < GAIN_MIN:
                break
            latid = int(CR[j])
            if latid in unit:
                continue
            if precision_l[latid] < PRECISION_FLOOR:
                continue
            fl = fire_diagpos_d[:, j]; ok = True
            for fm in member_fire.values():
                inter_m = int((fl & fm).sum()); union_m = int((fl | fm).sum())
                if union_m > 0 and inter_m / union_m >= JACCARD_MAX:
                    ok = False; break
            if not ok:
                continue
            col = fl[H].astype(np.float64)
            _, lo, _ = bootstrap_ci(col, B=B_GATE)
            if lo <= 0:
                continue
            best_l, best_gain, best_j = latid, float(gains[j]), j; break
        if best_l is None:
            break
        unit.append(best_l); member_fire[best_l] = fire_diagpos_d[:, best_j]
        covered = fire_diagpos_d[:, best_j]
        edges.append({"anchor": anchor, "absorber": best_l,
            "specializes": str(subs_sorted[int(subctx_arg_cr[best_j])]) if subs_sorted else None,
            "marginal_gain": float(best_gain), "jaccard": float(jaccard_cr[best_j]),
            "subctx_precision": float(subctx_prec_cr[best_j])})
        H = H & ~covered
        logger.info(f"    + absorber {best_l} specializes={edges[-1]['specializes']} "
                    f"gain={best_gain:.3f} remaining_holes={int(H.sum())}")
    recovered = len(unit) - 1
    logger.info(f"  K-track unit members={unit} | recovered_absorbers={recovered}")
    admission = admission_check(unit, anchor, cr_idx, A_on, A_off, lat, meta, rt, role, fold,
                                pair_id, sub, width)
    pos_tr = np.where(corp_train & (label == 1))[0]; neg_tr = np.where(corp_train & (label == 0))[0]
    mean_pos = np.asarray(lat[pos_tr].mean(0)).ravel(); mean_neg = np.asarray(lat[neg_tr].mean(0)).ravel()
    attribution = np.abs(mean_pos - mean_neg); attr_rank = np.argsort(-attribution)
    g_pool = attr_rank[:20].tolist(); h_pool = attr_rank[:max(len(unit), 1)].tolist()
    logger.info(f"  (g) top-20 pool head={g_pool[:5]} | (h) top-{len(h_pool)} pool={h_pool}")
    lat_diag = lat[diag_rows].tocsc(); diag_label = label[diag_rows]; diag_sub = sub[diag_rows]
    pos_in_diag = diag_label == 1
    probe_score_diag = probe.decision_function(resid[diag_rows].astype(np.float32)).astype(np.float64)
    def pool_score(pool):
        if not pool:
            return np.zeros(len(diag_rows))
        sub_mat = np.asarray(lat_diag[:, pool].todense()); return sub_mat.max(1)
    scores = {"unit": pool_score(unit), "anchor": pool_score([anchor]), "g": pool_score(g_pool),
              "h": pool_score(h_pool), "dense_probe": probe_score_diag}
    overall_recall_raw = {k: float((scores[k][pos_in_diag] > 0.0).mean()) for k in scores}
    R_match = min(overall_recall_raw["unit"], overall_recall_raw["g"], overall_recall_raw["h"],
                  overall_recall_raw["dense_probe"], overall_recall_raw["anchor"])
    R_match = max(R_match, 0.05)
    taus = {k: match_threshold(scores[k][pos_in_diag], R_match) for k in scores}
    fires_raw = {k: (scores[k] > 0.0) for k in scores}
    fires_matched = {k: (scores[k] >= taus[k]) for k in scores}
    fp_matched = {k: float(fires_matched[k][~pos_in_diag].mean()) if (~pos_in_diag).any() else 0.0 for k in scores}
    logger.info(f"  overall recall (>0 rule): {{{', '.join(f'{k}:{v:.3f}' for k,v in overall_recall_raw.items())}}} | R_match={R_match:.3f}")
    logger.info(f"  FP rate (matched): {{{', '.join(f'{k}:{v:.3f}' for k,v in fp_matched.items())}}}")
    sliced = {}; pvals_g_matched, pvals_h_matched = {}, {}; absorbed_subs = []
    for s in sorted([x for x in set(diag_sub[pos_in_diag].tolist()) if x is not None]):
        m = pos_in_diag & (diag_sub == s); ns = int(m.sum())
        elig = (str(s) in eligible) and ns >= N_MIN_ELIGIBLE
        rec_raw = {k: float(fires_raw[k][m].mean()) for k in scores}
        rec_mat = {k: float(fires_matched[k][m].mean()) for k in scores}
        is_absorbed = rec_raw["anchor"] < (overall_recall_raw["anchor"] - 0.10)
        ent = {"n": ns, "eligible": bool(elig), "absorbed": bool(is_absorbed),
               "recall_raw": rec_raw, "recall_matched": rec_mat}
        if elig:
            for comp, pool_key in (("g", "g"), ("h", "h")):
                a = fires_matched["unit"][m]; b = fires_matched[pool_key][m]
                diff, lo, hi = paired_diff_ci(a, b, B=B_PAIRED); p = mcnemar_exact(a, b)
                ent[f"unit_minus_{comp}_matched"] = {"diff": diff, "ci_lo": lo, "ci_hi": hi, "mcnemar_p": p}
                ar = fires_raw["unit"][m]; br = fires_raw[pool_key][m]
                d2, l2, h2 = paired_diff_ci(ar, br, B=B_PAIRED)
                ent[f"unit_minus_{comp}_raw"] = {"diff": d2, "ci_lo": l2, "ci_hi": h2, "mcnemar_p": mcnemar_exact(ar, br)}
                if comp == "g":
                    pvals_g_matched[str(s)] = p
                else:
                    pvals_h_matched[str(s)] = p
            if is_absorbed:
                absorbed_subs.append(str(s))
        sliced[str(s)] = ent
    holm_g = holm(pvals_g_matched); holm_h = holm(pvals_h_matched)
    for s in sliced:
        if s in holm_g:
            sliced[s].setdefault("unit_minus_g_matched", {})["holm_p"] = holm_g[s]
        if s in holm_h:
            sliced[s].setdefault("unit_minus_h_matched", {})["holm_p"] = holm_h[s]
    c3_confirmed = False
    for s in absorbed_subs:
        e = sliced[s]; gci = e.get("unit_minus_g_matched", {}); hci = e.get("unit_minus_h_matched", {})
        if recovered >= 1 and ((gci.get("ci_lo", -1) > 0) or (hci.get("ci_lo", -1) > 0)):
            c3_confirmed = True; break
    kg = formfree_edge_agreement(edges, anchor, CR, lat, resid, diagpos_rows, sub_diagpos,
                                 anchor_fire, d_p_unit, width)
    recall_vals = list(parent_probe_recall_by_sub.values())
    uniformity = {"min_probe_recall": float(np.min(recall_vals)) if recall_vals else 0.0,
                  "max_probe_recall": float(np.max(recall_vals)) if recall_vals else 0.0,
                  "std_probe_recall": float(np.std(recall_vals)) if recall_vals else 0.0}
    result = {"gate_decision": "PASS" if gate_pass else "FAIL", "gate_G1_recall_ok": bool(g1),
        "gate_G2_absorber_exists": bool(g2), "anchor_latent": anchor,
        "anchor_precision": float(precision_l[anchor]), "anchor_neg_fire_rate": float(neg_fire_rate[anchor]),
        "anchor_recall_contentflip": anchor_recall_cf, "anchor_recall_corpus": anchor_recall_corp,
        "n_content_responsive": int(len(cr_idx)), "n_pairs_train": Npair,
        "n_diag_positives": int(len(diagpos_rows)), "holes_corpus": H0, "holes_contentflip": int(holes_cf.sum()),
        "non_triviality_passing_absorbers": passing, "threshold_sensitivity": threshold_sensitivity,
        "k_track_unit": unit, "recovered_absorber_count": recovered, "kg_edges": edges,
        "admission": admission, "g_pool": g_pool, "h_pool": h_pool,
        "overall_recall_raw": overall_recall_raw,
        "overall_recall_matched": {k: float(fires_matched[k][pos_in_diag].mean()) for k in scores},
        "fp_rate_matched": fp_matched, "matched_recall_target": R_match, "sliced_recall": sliced,
        "absorbed_subcontexts": absorbed_subs, "c3_confirmed": bool(c3_confirmed),
        "parent_probe_recall_by_subcontext": parent_probe_recall_by_sub,
        "parent_probe_overall_recall": overall_probe_recall, "probe_recall_uniformity": uniformity,
        "kg_agreement": kg, "eligible_subcontexts": eligible}
    np.savez_compressed(RESULTS_DIR / f"arrays_{name}_w{width}.npz",
        anchor=np.array([anchor]), unit=np.array(unit), attr_rank=attr_rank[:200],
        attribution=attribution, content_responsive=cr_idx, jaccard_cr=jaccard_cr,
        subctx_prec_cr=subctx_prec_cr, hole_cov_cr=hole_cov_cr, d_p=d_p_unit)
    ctx = {"diag_rows": diag_rows, "fires_matched": fires_matched, "diagpos_rows": diagpos_rows,
           "scores": scores, "unit": unit, "anchor": anchor, "g_pool": g_pool, "h_pool": h_pool,
           "label": label, "sub": sub, "meta": meta}
    return result, ctx

## Run the pipeline

In [ ]:
t0 = time.time()
result, ctx = analyze_hierarchy(HIERARCHY, rows, lat, resid, get_eligible(HIERARCHY), WIDTH)
logger.info(f"DONE in {time.time()-t0:.1f}s | gate={result['gate_decision']} "
            f"recovered={result['recovered_absorber_count']} c3_confirmed={result['c3_confirmed']} "
            f"absorbed={result['absorbed_subcontexts']}")

## Results & visualization

The table shows, per **eligible** sub-context, the recall of each detector under the `>0` JumpReLU
firing rule:
`unit` (anchor + K-track absorbers) · `anchor` (raw single latent) · `(g)` top-20 oracle pool ·
`(h)` count-matched pool · `dense` (non-SAE residual probe).

**How to read the demo outcome.** The **gate PASSES** — numeric absorption is detected beyond
spelling (the anchor leaves holes a sub-context-precise latent covers, and K-track recovers a
named specialist). The **dense non-SAE probe reaches recall 1.0** — the honest "simple baselines
can match raw-SAE latents" finding the paper reports. At 100 rows the paired *unit − (g)/(h)*
comparison is under-powered, so **C3 is not confirmed at demo scale** (the full run uses
`N_MIN_ELIGIBLE=150` over thousands of rows). Latent IDs differ from the paper because the anchor
and absorbers are re-derived on this subset.

In [ ]:
# ---- summary ----
print(f"VERDICT for '{HIERARCHY}':  gate={result['gate_decision']}  "
      f"c3_confirmed={result['c3_confirmed']}")
print(f"anchor latent       : {result['anchor_latent']}  "
      f"(precision={result['anchor_precision']:.3f}, neg-fire={result['anchor_neg_fire_rate']:.3f})")
print(f"anchor recall        : content-flip={result['anchor_recall_contentflip']:.3f}  "
      f"corpus={result['anchor_recall_corpus']:.3f}")
print(f"content-responsive   : {result['n_content_responsive']} latents | "
      f"holes(corpus)={result['holes_corpus']}/{result['n_diag_positives']}")
print(f"K-track unit         : {result['k_track_unit']}  "
      f"(recovered {result['recovered_absorber_count']} absorber(s))")
for e in result["kg_edges"]:
    print(f"   edge anchor {e['anchor']} -> absorber {e['absorber']} "
          f"specializes '{e['specializes']}'  gain={e['marginal_gain']:.3f}")
print(f"absorbed sub-contexts: {result['absorbed_subcontexts']}")
print(f"admission signature-K: pass={result['admission']['signature_K_pass']} "
      f"(gainK={result['admission']['gainK']:.3f} vs null95={result['admission']['null_95']:.3f})")
print(f"KG-agreement (Phase7): {result['kg_agreement'].get('note')}")

print("\n--- sliced recall (>0 rule) on eligible sub-contexts ---")
det = ["unit", "anchor", "g", "h", "dense_probe"]
hdr = "sub-context  n  absorbed | " + "  ".join(f"{d:>11s}" for d in det)
print(hdr); print("-" * len(hdr))
rowtab = []
for s, e in result["sliced_recall"].items():
    if not e["eligible"]:
        continue
    rr = e["recall_raw"]
    print(f"{s:>11s} {e['n']:>2d} {str(e['absorbed']):>8s} | " +
          "  ".join(f"{rr[d]:>11.3f}" for d in det))
    rowtab.append((s, e["absorbed"], [rr[d] for d in det]))

In [ ]:
# ---- grouped bar chart: per-sub-context recall by detector + FP-rate panel ----
labels = [r[0] + ("\n(absorbed)" if r[1] else "") for r in rowtab]
x = np.arange(len(rowtab)); w = 0.16
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for j, d in enumerate(det):
    ax1.bar(x + (j - 2) * w, [r[2][j] for r in rowtab], w, label=d, color=colors[j])
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel("recall (>0 firing rule)"); ax1.set_ylim(0, 1.08)
ax1.set_title(f"[{HIERARCHY}] detector recall per eligible sub-context")
ax1.legend(fontsize=8, ncol=2); ax1.grid(axis="y", alpha=0.3)

fp = result["fp_rate_matched"]
ax2.bar(range(len(det)), [fp[d] for d in det], color=colors)
ax2.set_xticks(range(len(det))); ax2.set_xticklabels(det, rotation=20)
ax2.set_ylabel("false-positive rate (matched)")
ax2.set_title("FP rate on diagnostic negatives\n(unit is near-zero / clean)")
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "demo_recall.png", dpi=110); plt.show()
print("gate:", result["gate_decision"], "| unit FP:", fp["unit"], "| dense-probe FP:", fp["dense_probe"])